<h1 style="text-align:center; color:#00F1FA;">MCQ Creator App</h1>
<p style="text-align:center; color:#00F1FA;">Loads PDF documents, generates embeddings, stores them in Pinecone, retrieves relevant content, and structures the output as multiple choice questions.</p>

<h2 style="color:#4BFA00;">Table of Contents</h2>
<ul style="color:#4BFA00;">
    <li>Install and Import Dependencies</li>
    <li>Load Documents</li>
    <li>Transform Documents</li>
    <li>Generate Text Embeddings</li>
    <li>Vector Store - Pinecone</li>
    <li>Retrieve Answers</li>
    <li>Structure the Output</li>
</ul>

<h2 style="color:#FA00EE;">Install Libraries</h2>
<p style="color:#FA00EE;">Uncomment and run the lines below if any package is not installed in your environment.</p>

In [1]:
# Uncomment and run if packages are not installed
#!pip install openai==1.14.2
#!pip install langchain==0.1.13
#!pip install unstructured==0.12.3
#!pip install tiktoken==0.5.2
#!pip install pinecone-client==3.2.0
#!pip install pypdf==4.1.0
#!pip install sentence-transformers==2.5.1

<h2 style="color:#FA00EE;">Import Dependencies</h2>
<p style="color:#FA00EE;">Importing all required libraries for document loading, text splitting, embeddings, vector storage, and question answering.</p>

In [2]:
import os
import re
import json
import openai

# Document loading and splitting
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings

# Vector store
from langchain_community.vectorstores import Pinecone

# LLM
from langchain_openai import OpenAI

# QA chain
from langchain_classic.chains.question_answering import load_qa_chain

# Output parsing
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, HumanMessagePromptTemplate
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema

/Users/kpitsiakkos/Documents/Langchain-and-Langgraph-Specialization/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Library/Frameworks/Python.framework/Versions/3

In [3]:
from dotenv import load_dotenv

# Load keys from .env file
load_dotenv(dotenv_path="/Users/kpitsiakkos/Documents/Langchain-and-Langgraph-Specialization/Multiple_Choice_Question_Creator_App/.env")

# Verify keys loaded
print("OPENAI KEY FOUND:", bool(os.getenv("OPENAI_API_KEY")))
print("PINECONE KEY FOUND:", bool(os.getenv("PINECONE_API_KEY")))
print("HUGGINGFACE KEY FOUND:", bool(os.getenv("HUGGINGFACEHUB_API_TOKEN")))

OPENAI KEY FOUND: True
PINECONE KEY FOUND: False
HUGGINGFACE KEY FOUND: False


<h2 style="color:#00F1FA;">Load Documents</h2>
<p style="color:#00F1FA;">Reads all PDF files from the Docs directory using PyPDFDirectoryLoader. Each PDF page is loaded as a separate document object.</p>

In [4]:
!pip install pypdf

In [5]:
# Function to load all PDFs from a directory
def load_docs(directory):
    loader = PyPDFDirectoryLoader(directory)
    documents = loader.load()
    return documents

# Updated path to the correct directory
directory = '/Users/kpitsiakkos/Documents/Langchain-and-Langgraph-Specialization/Multiple_Choice_Question_Creator_App/'
documents = load_docs(directory)

# Check total number of pages loaded across all PDFs
print(f"Total pages loaded: {len(documents)}")

Total pages loaded: 5


<h2 style="color:#00F1FA;">Transform Documents</h2>
<p style="color:#00F1FA;">Splits the loaded documents into smaller chunks using RecursiveCharacterTextSplitter. This is necessary because LLMs have token limits and cannot process entire documents at once.</p>
<ul style="color:#00F1FA;">
    <li>chunk_size=1000 - each chunk contains up to 1000 characters</li>
    <li>chunk_overlap=20 - 20 characters overlap between consecutive chunks to preserve context at boundaries</li>
</ul>

In [6]:
# Function to split documents into chunks
def split_docs(documents, chunk_size=1000, chunk_overlap=20):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    docs = text_splitter.split_documents(documents)
    return docs

docs = split_docs(documents)

# Check total number of chunks created
print(f"Total chunks created: {len(docs)}")

Total chunks created: 13


<h2 style="color:#4BFA00;">Generate Text Embeddings</h2>
<p style="color:#4BFA00;">Converts each text chunk into a vector, a list of floating point numbers that represent the meaning of the text. Similar meanings produce similar vectors, which enables semantic search.</p>
<p style="color:#4BFA00;">We use OpenAI embeddings which produce high quality vectors and are compatible with Pinecone.</p>

In [7]:
# Initialize OpenAI embeddings
embeddings = OpenAIEmbeddings(openai_api_key=os.getenv("OPENAI_API_KEY"))

print("Embedding model loaded: OpenAI text-embedding-ada-002")

Embedding model loaded: OpenAI text-embedding-ada-002


In [8]:
# Test the embeddings model with a sample query
query_result = embeddings.embed_query("Hello Buddy")

# Show the number of dimensions in the vector
print(f"Vector dimensions: {len(query_result)}")

Vector dimensions: 1536


In [9]:
# Preview the raw vector values
query_result

[-0.014294981025159359,
 -0.0002794026513583958,
 -0.0014387911651283503,
 -0.034485336393117905,
 -0.02950296364724636,
 0.016486182808876038,
 -0.00267704832367599,
 -0.004356317222118378,
 -0.016264453530311584,
 -0.0023151086643338203,
 0.006573604419827461,
 -0.008797412738204002,
 0.004147631581872702,
 -0.00933869183063507,
 0.013812394812703133,
 -0.013616751879453659,
 0.03605047985911369,
 -0.017320925369858742,
 0.013577623292803764,
 0.010192999616265297,
 -0.013421108946204185,
 -0.00351831316947937,
 -3.3396871003787965e-05,
 0.002592269564047456,
 -0.007564862258732319,
 -0.015703611075878143,
 0.005451918113976717,
 -0.01678616926074028,
 0.00317919859662652,
 -0.028589962050318718,
 0.02423364482820034,
 0.015912296250462532,
 -0.011595107614994049,
 -0.005458439234644175,
 -0.026242246851325035,
 -0.024533631280064583,
 -0.014308024197816849,
 -0.011914658360183239,
 0.020816413685679436,
 -0.026242246851325035,
 0.021299000829458237,
 -0.005634518340229988,
 -0.00405

<h2 style="color:#FA00EE;">Vector Store - Pinecone</h2>
<p style="color:#FA00EE;">Stores all chunk embeddings in Pinecone, a cloud-based vector database. Unlike Chroma which runs in memory, Pinecone persists data in the cloud and can be reused across sessions.</p>
<p style="color:#FA00EE;">Pinecone allows true semantic search, it finds vectors that are closest in meaning to the query, not just exact keyword matches.</p>

In [13]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="/Users/kpitsiakkos/Documents/Langchain-and-Langgraph-Specialization/Multiple_Choice_Question_Creator_App/.env")

print("OPENAI KEY FOUND:", bool(os.getenv("OPENAI_API_KEY")))
print("PINECONE KEY FOUND:", bool(os.getenv("PINECONE_API_KEY")))

OPENAI KEY FOUND: True
PINECONE KEY FOUND: True


<h2 style="color:#00F1FA;">Retrieve Answers</h2>
<p style="color:#00F1FA;">When a question is asked, the retriever searches Pinecone for the most semantically similar chunks and returns them. The QA chain then passes those chunks to the LLM to generate a proper answer.</p>

In [14]:
# Function to find the top k most relevant chunks for a given query
def get_similiar_docs(query, k=2):
    similar_docs = index.similarity_search(query, k=k)
    return similar_docs

In [15]:
# Initialize OpenAI LLM for answering questions
llm = OpenAI(openai_api_key=os.getenv("OPENAI_API_KEY"))

# Load QA chain — 'stuff' passes all retrieved chunks directly to the LLM
chain = load_qa_chain(llm, chain_type="stuff")

/var/folders/tb/g1v2rhgn72b31g4hyg9skymh0000gn/T/ipykernel_87275/2052582959.py:5: LangChainDeprecationWarning: This class is deprecated. See the following migration guides for replacements based on `chain_type`:
stuff: https://python.langchain.com/docs/versions/migrating_chains/stuff_docs_chain
map_reduce: https://python.langchain.com/docs/versions/migrating_chains/map_reduce_chain
refine: https://python.langchain.com/docs/versions/migrating_chains/refine_chain
map_rerank: https://python.langchain.com/docs/versions/migrating_chains/map_rerank_docs_chain

See also guides on retrieval and question-answering here: https://python.langchain.com/docs/how_to/#qa-with-rag
  chain = load_qa_chain(llm, chain_type="stuff")


In [16]:
# Function to get an answer for a given query
def get_answer(query):
    relevant_docs = get_similiar_docs(query)
    print("Relevant chunks found:", relevant_docs)
    response = chain.run(input_documents=relevant_docs, question=query)
    return response

# Test with Cyprus and Greece questions
cyprus_query = "What is the capital of Cyprus?"
cyprus_answer = get_answer(cyprus_query)
print("Cyprus Answer:", cyprus_answer)

greece_query = "What is the currency of Greece?"
greece_answer = get_answer(greece_query)
print("Greece Answer:", greece_answer)

NameError: name 'index' is not defined